# Layer 2 — Quality Tests

Validates that all Layer 2 detection scripts produced correct and consistent outputs before Layer 3 (Root Cause Analysis) runs.

| # | Test | What it checks |
|---|------|----------------|
| 1 | Method A shape & Flag A1 | (8772, 18) shape · flag_a1=0 · STL=367 · WoW+MoM=1,602 |
| 2 | Method A recall = 1.0 | All 20 ground-truth anomaly days flagged · zero false negatives |
| 3 | Method B shape & score separation | 34 flagged days · flagged scores > 0 · non-flagged scores < 0 |
| 4 | Method C shape & CI coverage | 100% CI on revenue/orders/CVR · 94.5% on volatile avg_roas |
| 5 | Ensemble voting matrix | (8772, 31) · votes 0→6620, 1→1971, 2→166, 3→15 · confirmed=181 |
| 6 | Confirmed anomaly count & recall | (181, 25) · 68 unique dates · 12/20 GT days caught (Recall=0.600) |
| 7 | Severity distribution | HIGH=15 · MEDIUM=92 · LOW=74 |
| 8 | Key ground-truth events | Black Friday HIGH/3-votes/UP · Stockout MEDIUM/DOWN · Email spike MEDIUM/UP |
| 9 | Anomaly ID format | All 181 IDs match ANO-YYYYMMDD-{CODE} · all unique |
| 10 | SQLite table parity | All 6 tables present with correct row counts |

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import re
from pathlib import Path
import os

# Resolve project root whether notebook is run from scripts/ or project root
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'scripts':
    PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

DATA = PROJECT_ROOT / 'data'

print(f'Project root : {PROJECT_ROOT}')
print(f'Data folder  : {DATA}')

# Load all Layer 2 outputs once — shared across all tests
a = pd.read_csv(DATA / 'method_a_results.csv')
b = pd.read_csv(DATA / 'method_b_results.csv')
c = pd.read_csv(DATA / 'method_c_results.csv')
m = pd.read_csv(DATA / 'ensemble_voting_matrix.csv')
r = pd.read_csv(DATA / 'anomaly_results.csv')

print()
print('Files loaded:')
print(f'  method_a_results.csv       {a.shape}')
print(f'  method_b_results.csv       {b.shape}')
print(f'  method_c_results.csv       {c.shape}')
print(f'  ensemble_voting_matrix.csv {m.shape}')
print(f'  anomaly_results.csv        {r.shape}')

Project root : C:\Users\annes\OneDrive\Adidas Office One Drive - Personal Folders\Upskilling\2026\KPI Anomaly Detection Agent
Data folder  : C:\Users\annes\OneDrive\Adidas Office One Drive - Personal Folders\Upskilling\2026\KPI Anomaly Detection Agent\data



Files loaded:
  method_a_results.csv       (8772, 18)
  method_b_results.csv       (731, 9)
  method_c_results.csv       (2924, 15)
  ensemble_voting_matrix.csv (8772, 31)
  anomaly_results.csv        (181, 25)


---
## Test 1 — Method A Shape and Flag A1 = 0

Method A must produce 8,772 rows (12 KPIs × 731 days). Flag A1 (7-day rolling z-score) must fire zero times because the maximum absolute z-score in the feature table is 2.268 — below the 2.5 threshold. STL residual (Flag A2) is the primary signal.

Expected:
```
PASS  Method A shape: (8772, 18)
PASS  flag_a1 (z-score)   fires:     0  (7-day z never exceeded +-2.5 threshold)
PASS  flag_a2 (STL)       fires:   367
PASS  flag_a3 (WoW+MoM)   fires:  1602
PASS  method_a_flag total fires:  1886
```

In [2]:
assert a.shape == (8772, 18), f'Expected (8772, 18), got {a.shape}'
assert a['flag_a1'].sum() == 0, \
    f'flag_a1 should fire 0 times — max |z| in dataset is 2.268 (below +-2.5 threshold)'
assert a['flag_a2'].sum() == 367, \
    f'Expected 367 STL residual flags, got {a["flag_a2"].sum()}'
assert a['flag_a3'].sum() == 1602, \
    f'Expected 1,602 WoW+MoM combined flags, got {a["flag_a3"].sum()}'

print(f'PASS  Method A shape: {a.shape}')
print(f'PASS  flag_a1 (z-score)   fires: {a["flag_a1"].sum():>5}  (7-day z never exceeded +-2.5 threshold)')
print(f'PASS  flag_a2 (STL)       fires: {a["flag_a2"].sum():>5}')
print(f'PASS  flag_a3 (WoW+MoM)   fires: {a["flag_a3"].sum():>5}')
print(f'PASS  method_a_flag total fires: {a["method_a_flag"].sum():>5}')

PASS  Method A shape: (8772, 18)
PASS  flag_a1 (z-score)   fires:     0  (7-day z never exceeded +-2.5 threshold)
PASS  flag_a2 (STL)       fires:   367
PASS  flag_a3 (WoW+MoM)   fires:  1602
PASS  method_a_flag total fires:  1886


---
## Test 2 — Method A Recall = 1.0

Method A is the over-sensitive first pass. It must flag at least one KPI on every one of the 20 ground-truth anomaly days. Zero false negatives is the design requirement.

Expected:
```
PASS  Method A true positives : 20 / 20  (Recall = 1.000)
PASS  Method A false negatives: 0  (all anomaly days detected)
```

In [3]:
daily_flags = a.groupby('date')['method_a_flag'].any().reset_index()
gt          = a[['date', 'anomaly_flag']].drop_duplicates('date')
merged      = gt.merge(daily_flags, on='date')

tp = int(((merged['anomaly_flag'] == 1) &  merged['method_a_flag']).sum())
fn = int(((merged['anomaly_flag'] == 1) & ~merged['method_a_flag']).sum())

assert tp == 20, f'Expected 20 true positives (all anomaly days caught), got {tp}'
assert fn == 0,  f'Expected 0 false negatives (no missed anomaly days), got {fn}'

print(f'PASS  Method A true positives : {tp} / 20  (Recall = {tp/20:.3f})')
print(f'PASS  Method A false negatives: {fn}  (all anomaly days detected)')

PASS  Method A true positives : 20 / 20  (Recall = 1.000)
PASS  Method A false negatives: 0  (all anomaly days detected)


---
## Test 3 — Method B Shape and Score Separation

Method B must produce exactly 731 rows (one per day) and flag 34 days. A correctly fitted Isolation Forest must assign strictly positive anomaly scores to all flagged days and strictly negative scores to all non-flagged days — confirming a clean decision boundary at zero.

Expected:
```
PASS  Method B shape: (731, 9)
PASS  Flagged days: 34  (flag rate: 4.7%)
PASS  Min score (flagged):     0.0004
PASS  Max score (non-flagged): -0.0020
PASS  Score separation is clean -- no overlap at threshold 0
```

In [4]:
assert b.shape == (731, 9), f'Expected (731, 9), got {b.shape}'
assert b['method_b_flag'].sum() == 34, \
    f'Expected 34 flagged days, got {b["method_b_flag"].sum()}'
assert b.loc[ b['method_b_flag'], 'method_b_score'].min() > 0, \
    'All flagged days must have positive anomaly score'
assert b.loc[~b['method_b_flag'], 'method_b_score'].max() < 0, \
    'All non-flagged days must have negative anomaly score'

print(f'PASS  Method B shape: {b.shape}')
print(f'PASS  Flagged days: {int(b["method_b_flag"].sum())}  '
      f'(flag rate: {b["method_b_flag"].mean()*100:.1f}%)')
print(f'PASS  Min score (flagged):     {b.loc[ b["method_b_flag"],"method_b_score"].min():.4f}')
print(f'PASS  Max score (non-flagged): {b.loc[~b["method_b_flag"],"method_b_score"].max():.4f}')
print(f'PASS  Score separation is clean -- no overlap at threshold 0')

PASS  Method B shape: (731, 9)
PASS  Flagged days: 34  (flag rate: 4.7%)
PASS  Min score (flagged):     0.0004
PASS  Max score (non-flagged): -0.0020
PASS  Score separation is clean -- no overlap at threshold 0


---
## Test 4 — Method C Shape and Prediction Interval Coverage

Method C must produce 2,924 rows (4 Tier 1 KPIs × 731 days). A well-calibrated 95% CI should contain ~95% of non-anomaly days. `total_revenue_usd`, `n_orders`, and `conversion_rate` must achieve exactly 100% coverage — confirming the Prophet models are not over-flagging. `avg_roas` is expected to fall slightly below 100% due to its naturally wide value range.

Expected:
```
PASS  total_revenue_usd            CI coverage on non-anomaly days: 100.0%
PASS  n_orders                     CI coverage on non-anomaly days: 100.0%
PASS  conversion_rate              CI coverage on non-anomaly days: 100.0%
PASS  avg_roas                     CI coverage on non-anomaly days: 94.5%  (volatile KPI)
```

In [5]:
assert c.shape == (2924, 15), f'Expected (2924, 15), got {c.shape}'

for kpi, expected_cov in [
    ('total_revenue_usd', 100.0),
    ('n_orders',          100.0),
    ('conversion_rate',   100.0),
]:
    normal   = c[(c['kpi'] == kpi) & (c['anomaly_flag'] == 0)]
    coverage = (~normal['method_c_flag']).mean() * 100
    assert coverage == expected_cov, \
        f'{kpi}: CI coverage = {coverage:.1f}%, expected {expected_cov:.1f}%'
    print(f'PASS  {kpi:<28}  CI coverage on non-anomaly days: {coverage:.1f}%')

roas_normal   = c[(c['kpi'] == 'avg_roas') & (c['anomaly_flag'] == 0)]
roas_coverage = (~roas_normal['method_c_flag']).mean() * 100
assert 90 < roas_coverage < 100, \
    f'avg_roas CI coverage {roas_coverage:.1f}% outside expected range (90-100%)'
print(f'PASS  avg_roas                       CI coverage on non-anomaly days: {roas_coverage:.1f}%  '
      f'(volatile KPI)')

PASS  total_revenue_usd             CI coverage on non-anomaly days: 100.0%
PASS  n_orders                      CI coverage on non-anomaly days: 100.0%
PASS  conversion_rate               CI coverage on non-anomaly days: 100.0%
PASS  avg_roas                       CI coverage on non-anomaly days: 94.5%  (volatile KPI)


---
## Test 5 — Ensemble Voting Matrix Shape and Vote Distribution

The ensemble matrix must cover all 8,772 KPI-day pairs. 75.5% of pairs receive zero votes (normal baseline) and only 2.1% reach the confirmed threshold (votes >= 2).

Expected:
```
PASS  Ensemble matrix shape: (8772, 31)
PASS  votes=0  NORMAL    :  6620  (75.5%)
PASS  votes=1  WATCH     :  1971  (22.5%)
PASS  votes=2  CONFIRMED :   166  ( 1.9%)
PASS  votes=3  ALL AGREE :    15  ( 0.2%)
PASS  Total confirmed (votes >= 2): 181
```

In [6]:
assert m.shape == (8772, 31), f'Expected (8772, 31), got {m.shape}'
assert set(m['votes'].unique()).issubset({0, 1, 2, 3}), \
    'votes column must only contain values 0, 1, 2, 3'

vc = m['votes'].value_counts().sort_index()

assert vc[0] == 6620, f'Expected 6,620 rows with 0 votes, got {vc[0]}'
assert vc[1] == 1971, f'Expected 1,971 rows with 1 vote,  got {vc[1]}'
assert vc[2] == 166,  f'Expected   166 rows with 2 votes, got {vc[2]}'
assert vc[3] == 15,   f'Expected    15 rows with 3 votes, got {vc[3]}'
assert m['confirmed'].sum() == 181, \
    f'Expected 181 confirmed KPI-day pairs (votes >= 2), got {m["confirmed"].sum()}'

print(f'PASS  Ensemble matrix shape: {m.shape}')
print(f'PASS  votes=0  NORMAL    : {vc[0]:>5}  ({vc[0]/8772*100:.1f}%)')
print(f'PASS  votes=1  WATCH     : {vc[1]:>5}  ({vc[1]/8772*100:.1f}%)')
print(f'PASS  votes=2  CONFIRMED :  {vc[2]:>4}  ({vc[2]/8772*100:.1f}%)')
print(f'PASS  votes=3  ALL AGREE :   {vc[3]:>3}  ({vc[3]/8772*100:.1f}%)')
print(f'PASS  Total confirmed (votes >= 2): {m["confirmed"].sum()}')

PASS  Ensemble matrix shape: (8772, 31)
PASS  votes=0  NORMAL    :  6620  (75.5%)
PASS  votes=1  WATCH     :  1971  (22.5%)
PASS  votes=2  CONFIRMED :   166  (1.9%)
PASS  votes=3  ALL AGREE :    15  (0.2%)
PASS  Total confirmed (votes >= 2): 181


---
## Test 6 — Confirmed Anomaly Count and Recall

`anomaly_results.csv` must contain exactly 181 confirmed KPI-day records across 68 unique dates. 12 of the 20 ground-truth anomaly days must appear (Recall = 0.600). The 8 missed events were detected only by Method A — a known structural limitation of requiring >= 2 methods to agree.

Expected:
```
PASS  anomaly_results shape  : (181, 25)
PASS  Unique anomaly dates   : 68
PASS  GT anomaly dates caught: 12 / 20  (Recall = 0.600)
PASS  Missed days (fn)       : 8  (only Method A detected; B and C both missed)
```

In [7]:
assert r.shape == (181, 25), f'Expected (181, 25), got {r.shape}'
assert r['date'].nunique() == 68, \
    f'Expected 68 unique anomaly dates, got {r["date"].nunique()}'

tp_dates = r[r['anomaly_flag'] == 1]['date'].nunique()
assert tp_dates == 12, \
    f'Expected 12 ground-truth anomaly dates caught (Recall=0.600), got {tp_dates}'

print(f'PASS  anomaly_results shape  : {r.shape}')
print(f'PASS  Unique anomaly dates   : {r["date"].nunique()}')
print(f'PASS  GT anomaly dates caught: {tp_dates} / 20  (Recall = {tp_dates/20:.3f})')
print(f'PASS  Missed days (fn)       : {20 - tp_dates}  '
      f'(only Method A detected; B and C both missed)')

PASS  anomaly_results shape  : (181, 25)
PASS  Unique anomaly dates   : 68
PASS  GT anomaly dates caught: 12 / 20  (Recall = 0.600)
PASS  Missed days (fn)       : 8  (only Method A detected; B and C both missed)


---
## Test 7 — Severity Distribution

Across all 181 confirmed KPI-day records, the severity breakdown must match exactly. HIGH is reserved for Tier 1 KPIs where all 3 methods agree simultaneously. MEDIUM covers Tier 1 with 2-method agreement and all Tier 2. LOW covers all Tier 3 confirmations.

Expected:
```
PASS  HIGH   :  15  (Tier 1 -- all 3 methods agree)
PASS  MEDIUM :  92  (Tier 1 with 2 methods  |  Tier 2 confirmed)
PASS  LOW    :  74  (Tier 3 confirmed)
PASS  Total  : 181
```

In [8]:
sev = r['severity'].value_counts()

assert sev.get('HIGH',   0) == 15, f'Expected 15 HIGH,   got {sev.get("HIGH", 0)}'
assert sev.get('MEDIUM', 0) == 92, f'Expected 92 MEDIUM, got {sev.get("MEDIUM", 0)}'
assert sev.get('LOW',    0) == 74, f'Expected 74 LOW,    got {sev.get("LOW", 0)}'

print(f'PASS  HIGH   : {sev.get("HIGH",   0):>3}  (Tier 1 -- all 3 methods agree)')
print(f'PASS  MEDIUM : {sev.get("MEDIUM", 0):>3}  (Tier 1 with 2 methods  |  Tier 2 confirmed)')
print(f'PASS  LOW    : {sev.get("LOW",    0):>3}  (Tier 3 confirmed)')
print(f'PASS  Total  : {len(r):>3}')

PASS  HIGH   :  15  (Tier 1 -- all 3 methods agree)
PASS  MEDIUM :  92  (Tier 1 with 2 methods  |  Tier 2 confirmed)
PASS  LOW    :  74  (Tier 3 confirmed)
PASS  Total  : 181


---
## Test 8 — Key Ground-Truth Events Confirmed Correctly

Three specific labeled anomaly events must be confirmed with the correct severity, direction, and vote count — validating both upward spikes and downward drops are characterised accurately.

Expected:
```
PASS  2024-11-29  black_friday_spike       total_revenue_usd  [HIGH]    votes=3  UP    dev=+223.8%
PASS  2024-03-15  inventory_stockout       n_orders           [MEDIUM]  votes=2  DOWN  dev=-35.7%
PASS  2024-09-03  email_campaign_spike     conversion_rate    [MEDIUM]  votes=2  UP    dev=+65.8%
```

In [9]:
# Black Friday 2024-11-29: all 3 methods agree on revenue -> HIGH, votes=3, UP
bf = r[(r['date'] == '2024-11-29') & (r['kpi'] == 'total_revenue_usd')].iloc[0]
assert bf['severity']   == 'HIGH',   f'Black Friday revenue: expected HIGH,   got {bf["severity"]}'
assert int(bf['votes']) == 3,        f'Black Friday revenue: expected 3 votes, got {bf["votes"]}'
assert bf['direction']  == 'UP',     f'Black Friday revenue: expected UP,      got {bf["direction"]}'
print(f'PASS  2024-11-29  black_friday_spike       '
      f'total_revenue_usd  [{bf["severity"]}]  '
      f'votes={int(bf["votes"])}  {bf["direction"]}  '
      f'dev={bf["deviation_pct"]:+.1f}%')

# Inventory stockout 2024-03-15: n_orders drops -> confirmed MEDIUM, DOWN
sk = r[(r['date'] == '2024-03-15') & (r['kpi'] == 'n_orders')].iloc[0]
assert sk['severity']  == 'MEDIUM',  f'Stockout n_orders: expected MEDIUM, got {sk["severity"]}'
assert sk['direction'] == 'DOWN',    f'Stockout n_orders: expected DOWN,   got {sk["direction"]}'
print(f'PASS  2024-03-15  inventory_stockout        '
      f'n_orders           [{sk["severity"]}]  '
      f'votes={int(sk["votes"])}  {sk["direction"]}  '
      f'dev={sk["deviation_pct"]:+.1f}%')

# Email campaign spike 2024-09-03: conversion_rate surges -> confirmed MEDIUM, UP
em = r[(r['date'] == '2024-09-03') & (r['kpi'] == 'conversion_rate')].iloc[0]
assert em['severity']  == 'MEDIUM',  f'Email spike cvr: expected MEDIUM, got {em["severity"]}'
assert em['direction'] == 'UP',      f'Email spike cvr: expected UP,     got {em["direction"]}'
print(f'PASS  2024-09-03  email_campaign_spike      '
      f'conversion_rate    [{em["severity"]}]  '
      f'votes={int(em["votes"])}  {em["direction"]}  '
      f'dev={em["deviation_pct"]:+.1f}%')

PASS  2024-11-29  black_friday_spike       total_revenue_usd  [HIGH]  votes=3  UP  dev=+223.8%
PASS  2024-03-15  inventory_stockout        n_orders           [MEDIUM]  votes=2  DOWN  dev=-35.7%
PASS  2024-09-03  email_campaign_spike      conversion_rate    [MEDIUM]  votes=2  UP  dev=+65.8%


---
## Test 9 — Anomaly ID Format

Every confirmed anomaly must have an ID following the format `ANO-YYYYMMDD-{KPI_CODE}`. No blank, null, or malformed IDs are acceptable — `anomaly_id` is the primary key used by Layer 3 to look up and link events.

Expected:
```
PASS  All 181 anomaly IDs match format  ANO-YYYYMMDD-{CODE}
PASS  All anomaly IDs are unique
PASS  Sample IDs: ['ANO-20240315-REV', 'ANO-20240417-REV', 'ANO-20240618-REV']
```

In [10]:
pattern     = re.compile(r'^ANO-\d{8}-[A-Z]+$')
invalid_ids = r[~r['anomaly_id'].apply(lambda x: bool(pattern.match(str(x))))]

assert len(invalid_ids) == 0, \
    f'Found {len(invalid_ids)} anomaly IDs with invalid format'
assert r['anomaly_id'].nunique() == len(r), \
    f'Anomaly IDs are not unique — expected {len(r)}, got {r["anomaly_id"].nunique()} distinct'

print(f'PASS  All {len(r)} anomaly IDs match format  ANO-YYYYMMDD-{{CODE}}')
print(f'PASS  All anomaly IDs are unique')
print(f'PASS  Sample IDs: {r["anomaly_id"].head(3).tolist()}')

PASS  All 181 anomaly IDs match format  ANO-YYYYMMDD-{CODE}
PASS  All anomaly IDs are unique
PASS  Sample IDs: ['ANO-20240315-REV', 'ANO-20240417-REV', 'ANO-20240618-REV']


---
## Test 10 — SQLite Table Parity

All six tables written by Layers 1 and 2 must exist in `kpi_anomaly_detection.db` and their row counts must match the corresponding CSV files exactly.

Expected:
```
PASS  processed_kpis                   731 rows
PASS  method_a_results               8,772 rows
PASS  method_b_results                 731 rows
PASS  method_c_results               2,924 rows
PASS  anomaly_results                  181 rows
PASS  ensemble_voting_matrix          8,772 rows
```

In [11]:
conn = sqlite3.connect(DATA / 'kpi_anomaly_detection.db')

expected = {
    'processed_kpis':          731,
    'method_a_results':       8772,
    'method_b_results':        731,
    'method_c_results':       2924,
    'anomaly_results':         181,
    'ensemble_voting_matrix': 8772,
}

tables_in_db = [
    row[0] for row in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table'"
    ).fetchall()
]

for table, expected_rows in expected.items():
    assert table in tables_in_db, f'Missing SQLite table: {table}'
    actual = conn.execute(f'SELECT COUNT(*) FROM [{table}]').fetchone()[0]
    assert actual == expected_rows, \
        f"Table '{table}': expected {expected_rows:,} rows, got {actual:,}"
    print(f'PASS  {table:<30}  {actual:>6,} rows')

conn.close()

PASS  processed_kpis                     731 rows
PASS  method_a_results                 8,772 rows
PASS  method_b_results                   731 rows
PASS  method_c_results                 2,924 rows
PASS  anomaly_results                    181 rows
PASS  ensemble_voting_matrix           8,772 rows


---
## Summary

In [12]:
tests = [
    ('Test 1',  'Method A Shape and Flag A1 = 0'),
    ('Test 2',  'Method A Recall = 1.0'),
    ('Test 3',  'Method B Shape and Score Separation'),
    ('Test 4',  'Method C Shape and CI Coverage'),
    ('Test 5',  'Ensemble Voting Matrix'),
    ('Test 6',  'Confirmed Anomaly Count and Recall'),
    ('Test 7',  'Severity Distribution'),
    ('Test 8',  'Key Ground-Truth Events Confirmed Correctly'),
    ('Test 9',  'Anomaly ID Format'),
    ('Test 10', 'SQLite Table Parity'),
]

print('=' * 55)
print('Layer 2 Quality Test Results')
print('=' * 55)
for num, name in tests:
    print(f'  PASS  {num:<8} -- {name}')
print('=' * 55)
print(f'  All 10 tests passed -- Layer 2 output is valid.')
print(f'  anomaly_results.csv is ready for Layer 3.')
print('=' * 55)

Layer 2 Quality Test Results
  PASS  Test 1   -- Method A Shape and Flag A1 = 0
  PASS  Test 2   -- Method A Recall = 1.0
  PASS  Test 3   -- Method B Shape and Score Separation
  PASS  Test 4   -- Method C Shape and CI Coverage
  PASS  Test 5   -- Ensemble Voting Matrix
  PASS  Test 6   -- Confirmed Anomaly Count and Recall
  PASS  Test 7   -- Severity Distribution
  PASS  Test 8   -- Key Ground-Truth Events Confirmed Correctly
  PASS  Test 9   -- Anomaly ID Format
  PASS  Test 10  -- SQLite Table Parity
  All 10 tests passed -- Layer 2 output is valid.
  anomaly_results.csv is ready for Layer 3.
